# Notebook 08 — Recommandations Copilote

Objectif : transformer le backlog priorisé et les scores business
en **recommandations actionnables**, sans décision automatique.

Ce notebook respecte la gouvernance :
- pas de décision IA
- règles explicites
- recommandations auditables


In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path('outputs')


## 1) Charger les entrées

On utilise :
- `block6_business_score.csv` (impact business)
- `block3_hotspots.csv` (contextes dominants)


In [ ]:
paths = {
    'business': OUT / 'block6_business_score.csv',
    'hotspots': OUT / 'block3_hotspots.csv'
}

missing = [k for k,v in paths.items() if not v.exists()]
if missing:
    raise FileNotFoundError(f'Manquant dans outputs/: {missing}')

df_business = pd.read_csv(paths['business'])
df_hotspots = pd.read_csv(paths['hotspots'])

df_business.head()

## 2) Règles métier explicites

Ces règles sont **lisibles, modifiables, auditables**.


In [ ]:
RULES = [
    {
        'if': lambda r: r['business_score'] > 0.7,
        'then': 'Action prioritaire (P1)'
    },
    {
        'if': lambda r: 0.4 <= r['business_score'] <= 0.7,
        'then': 'Action importante (P2)'
    },
    {
        'if': lambda r: r['business_score'] < 0.4,
        'then': 'Surveillance / amélioration continue (P3)'
    }
]

def apply_rules(row):
    for rule in RULES:
        if rule['if'](row):
            return rule['then']
    return 'Non classé'


## 3) Générer les recommandations


In [ ]:
df = df_business.copy()
df['priority_label'] = df.apply(apply_rules, axis=1)

df[['motif','business_score','priority_label']].head(10)

## 4) Lier avec les hotspots contextuels


In [ ]:
recos = df.merge(
    df_hotspots,
    on='motif',
    how='left'
)

recos.head(5)

## 5) Export final (copilote)


In [ ]:
OUT_FILE = OUT / 'block8_recommendations.csv'
recos.to_csv(OUT_FILE, index=False)
print('Exporté :', OUT_FILE)